# Where Clauses

Most questions in GIS are about some of the features in a dataset, not all of them: the libraries among all of the city's facilities, or the streets with a speed limit above 35 miles per hour. In M3, every tool you ran worked on a whole dataset. A where clause describes which features you want, so that a tool, and later a cursor, works on only those features. In this notebook, you will write where clauses in Python and test them with the Select tool.

🎯 What You'll Learn

* Set up ArcPy and a copy of the geodatabase
* Read a where clause as SQL and name its parts
* Use the Select tool to write the features that match a where clause to a new feature class
* Write where clauses with the most common SQL operators
* Build where clauses from variables with f-strings

Work through each numbered section: a demo shows how something works, then a ✏️ **Practice** prompt below it gives you a chance to try it yourself.

### ⚙️ Before you start: select the kernel

Check that the kernel in the top-right corner of the notebook is set to `arcgispro-py3`. If it is not, click the kernel name (or **Select Kernel**) and select `arcgispro-py3`. If it is not listed, refer back to `jupyter-notebook-basics.ipynb`.

---

### 1. Setup: ArcPy, folders, and a copy of the geodatabase

This notebook will work on a copy of `Austin.gdb` instead of the original in `austin_data`. The next cell creates the `outputs\m4` folder, copies `Austin.gdb` into it, and sets the environment settings so that every step in this notebook refers to the copy. The original data in `austin_data` remains untouched.

The cell copies `Austin.gdb` only if the copy does not exist yet. Rerunning the cell keeps the copy and every change you made to it. To start over with a fresh copy, follow the commented-out lines in the cell.

⚠️ **Important**: If your working folder is not `C:\workspace\gist`, update `working_folder` below.

In [ ]:
# Import ArcPy and os
import arcpy
import os

# Store the path to your working folder. Update this path to match the
# working folder you created in course-data-setup.ipynb
working_folder = r"C:\workspace\gist"
print(f"working_folder: {working_folder}")

# Build the path to the source geodatabase. It is only read, never changed
source_gdb = os.path.join(working_folder, "austin_data", "Austin.gdb")
print(f"Austin.gdb exists: {arcpy.Exists(source_gdb)}")

# Build the path to the outputs folder for this module, and create it if
# it does not exist yet
output_folder = os.path.join(working_folder, "outputs", "m4")
os.makedirs(output_folder, exist_ok=True)

# Build the path to the copy for this notebook
gdb_path = os.path.join(output_folder, "Austin.gdb")
print(f"gdb_path: {gdb_path}")

# Copy Austin.gdb only if the copy does not exist yet, so rerunning this
# cell keeps the changes you made to the copy
if not arcpy.Exists(gdb_path):
    arcpy.management.Copy(source_gdb, gdb_path)
    print("\tCopied Austin.gdb")
else:
    print("\tThe copy already exists")

    # To start over with a fresh copy, uncomment the next three lines and
    # rerun this cell. This deletes the copy and every change you made to
    # it, then copies Austin.gdb again
    # arcpy.management.Delete(gdb_path)
    # arcpy.management.Copy(source_gdb, gdb_path)
    # print("\tDeleted the copy and copied Austin.gdb again")

# Set the workspace to the copy, and allow tools to replace outputs
arcpy.env.workspace = gdb_path
arcpy.env.overwriteOutput = True
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

⚠️ **Important**: If the Delete tool raises an exception saying the geodatabase is locked, another program is using the copy. Close ArcGIS Pro if it is open, then restart the kernel and rerun the cells in this section from the top.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 2. SQL and the parts of a where clause

In ArcGIS Pro, the **Select By Attributes** tool lets you build a query by choosing a field, an operator, and a value from drop-down lists, or by typing the query yourself in its SQL view. ArcPy tools take the same query, written as text.

**SQL**, short for Structured Query Language, is the standard language for asking questions of the tables in a database. In a table, each row is a **record**, one feature or one item, and each column is a field. A **query** is a request for the records that meet a condition.

A complete SQL query reads `SELECT * FROM table_name WHERE condition`. ArcGIS supplies the `SELECT * FROM table_name` part for you, so ArcPy tools ask only for the condition that follows the keyword `WHERE`. That condition is called a **where clause**. For more information, see the [SQL reference for query expressions used in ArcGIS](https://doc.esri.com/en/arcgis-pro/latest/help/mapping/navigation/sql-reference-for-elements-used-in-query-expressions.html).

The simplest where clause has three parts:

| Part | Example | What it does |
| --- | --- | --- |
| Field | `STREET_TYP` | the field (or column) whose values are tested |
| Operator | `=` | how each value is compared |
| Value | `'DR'` | what each value is compared to |

The clause `STREET_TYP = 'DR'` matches every record whose `STREET_TYP` value is `DR`.

In Python, a where clause is stored as a string. The double quotes mark the start and end of the Python string, and everything inside them is SQL. Python does not check the SQL inside the string. ArcGIS checks it only when a tool uses the where clause, which is why a mistake in a where clause shows up as a tool error instead of a Python error.

In [ ]:
# Example 1 - Store a where clause as a string

# The double quotes belong to Python. Everything between them, including
# the single quotes, is SQL
street_type_clause = "STREET_TYP = 'DR'"
print(f"street_type_clause: {street_type_clause}")

# A where clause is an ordinary Python string
print(f"type(street_type_clause): {type(street_type_clause)}")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 3. Select tool

The **Select** tool, in the Analysis Tools toolbox, writes only the features that match a where clause to a new feature class. Its third parameter, `where_clause`, takes the clause as a string. For its syntax, parameters, and code samples, see the [Select tool documentation](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/analysis/select.html).

In this notebook, you will use the Select tool and the Get Count tool to test each where clause you write. The Select tool writes the matching features to a new feature class, and the Get Count tool counts the features in that output. If the count is what you expect, the where clause does what you meant.

The workspace is the copy of `Austin.gdb`, so an output given as a name only is written to the copy.

In [ ]:
# Example 1 - Select the 'DR' streets from the streets feature class

# Variable stores the full path of the streets feature class in the
# Transportation feature dataset. We will use variable gdb_path from Section 1
streets_fc = os.path.join(gdb_path, "Transportation", "streets")
print(f"streets_fc: {streets_fc}")

# Count every feature in the input, and convert the count to an integer
street_count = int(arcpy.management.GetCount(streets_fc)[0])
print(f"All streets: {street_count}")

# We will use variable street_type_clause from Section 2
print(f"street_type_clause: {street_type_clause}")

# Write only the features that match the where clause to a new feature class
select_result = arcpy.analysis.Select(
    in_features=streets_fc,
    out_feature_class="dr_streets",
    where_clause=street_type_clause
)

# Count the features in the output
dr_count = int(arcpy.management.GetCount(select_result)[0])
print(f"DR streets: {dr_count}")

⚠️ **Important**: A where clause can be valid SQL and still ask the wrong question. No error is raised, and the count looks believable, but it is wrong. This is common in code written by AI tools, which often guess field names and values without seeing the data. To catch it, check every where clause you write in this notebook:

1. Print the where clause before using it.
2. Check the count against something you know, such as the total number of features.
3. Enter the printed where clause in the SQL view of Select By Attributes in ArcGIS Pro and look at the records it selects.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 4. SQL operators

An **operator** is a symbol or keyword that performs an operation, such as comparing two values. The simplest SQL operator is **equal to** `=`, which tests whether two values are equal. SQL uses a single `=`, whereas Python uses `==`.

A **literal value** is a value typed directly into the clause and is used for comparison. Note the following:

* String values go inside **single** quotes: `STREET_TYP = 'DR'`.
* Numbers do not use quotes: `SPEED_LIMI = 35`.
* Field names do not use quotes: `SPEED_LIMI`, not `'SPEED_LIMI'`. A field name inside single quotes is a string value, not a field.
* String comparisons are **case sensitive**: `'DR'` and `'dr'` are different values. A value in the wrong case is valid SQL, so no error is raised. The clause simply matches nothing.

In [ ]:
# Example 1 - Equal to (=) with a string value

# The value is a string, so it goes inside single quotes
blvd_clause = "STREET_TYP = 'BLVD'"
blvd_result = arcpy.analysis.Select(
    streets_fc, "blvd", blvd_clause
)
print(f"{blvd_clause}: {arcpy.management.GetCount(blvd_result)[0]}")

# The stored value is 'BLVD', so 'blvd' matches nothing, and no error
# is raised
wrong_case_clause = "STREET_TYP = 'blvd'"
wrong_case_result = arcpy.analysis.Select(
    streets_fc, "wrong_case_blvd", wrong_case_clause
)
print(f"{wrong_case_clause}: "
      f"{arcpy.management.GetCount(wrong_case_result)[0]}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 4a] Use the Select tool and an appropriate where clause to select all
# post offices in the facilities feature class, and print the number of
# features in the output.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 5. Other comparison operators: `>`, `<`, `>=`, `<=`, `<>`

**Comparison operators** compare a field value to another value. They work the same way as in Python, except that SQL writes not equal to as `<>`, where Python uses `!=`.

| Comparison | Python | SQL |
| --- | --- | --- |
| equal to | `==` | `=` |
| not equal to | `!=` | `<>` |
| greater than | `>` | `>` |
| less than | `<` | `<` |
| greater than or equal to | `>=` | `>=` |
| less than or equal to | `<=` | `<=` |

In [ ]:
# Example 1 - Greater than (>) with a number

# The value is a number, so it is not quoted
fast_clause = "SPEED_LIMI > 35"
print(f"fast_clause: {fast_clause}")

# Write only the streets that match the where clause to a new feature class
fast_result = arcpy.analysis.Select(streets_fc, "fast_streets", fast_clause)

# Count the features in the output
fast_count = int(arcpy.management.GetCount(fast_result)[0])
print(f"Streets with a speed limit above 35: {fast_count}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 5a] Use the Select tool and an appropriate where clause to select all
# census blocks in the blockpop feature class with a 2010 population of
# 100 or less, and print the number of features in the output.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 6. Logical operators: `AND`, `OR`, `NOT`

**Logical operators** combine or reverse conditions, the same way `and`, `or`, and `not` do in Python. SQL writes them in capital letters by convention:

* `AND` matches a record only when both conditions are true.
* `OR` matches a record when at least one condition is true.
* `NOT` placed before a condition matches the records for which the condition is false.

When a clause mixes `AND` and `OR`, put **parentheses** `()` around the conditions that belong together, so they are evaluated first: `(STREET_TYP = 'DR' OR STREET_TYP = 'LN') AND SPEED_LIMI > 30`. Without the parentheses, the clause still runs, but it can answer a different question.

In [ ]:
# Example 1 - AND combines two conditions

# Both conditions must be true: drives with a speed limit above 30
drive_clause = "STREET_TYP = 'DR' AND SPEED_LIMI > 30"
print(f"drive_clause: {drive_clause}")

# Write only the streets that match both conditions to a new feature class
drive_result = arcpy.analysis.Select(streets_fc, "fast_drives", drive_clause)

# Count the features in the output
drive_count = int(arcpy.management.GetCount(drive_result)[0])
print(f"Drives with a speed limit above 30: {drive_count}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 6a] Use the Select tool and an appropriate where clause to select all
# restrooms in the restrooms feature class that are ADA compliant or have
# a flushing restroom and are not in 'Emma Long' park, and print the
# number of features in the output.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 7. Matching a list of values: `IN`

SQL's `IN` operator matches a record when the field value is one of the values listed inside the parentheses. A clause with several `OR` conditions on the same field is easier to read as `IN`.

In [ ]:
# Example 1 - IN matches any one of the listed values

# Same as STREET_TYP = 'DR' OR STREET_TYP = 'LN' OR STREET_TYP = 'CT'
in_clause = "STREET_TYP IN ('DR', 'LN', 'CT')"
print(f"in_clause: {in_clause}")

# Write only the streets that match one of the values to a new feature
# class
in_result = arcpy.analysis.Select(streets_fc, "dr_ln_ct_streets", in_clause)

# Count the features in the output
in_count = int(arcpy.management.GetCount(in_result)[0])
print(f"Drives, lanes, and courts: {in_count}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 7a] Use the Select tool and an appropriate where clause to select all
# watersheds in the watersheds feature class whose receiving water bodies
# (RECV_WATER) are Blanco River or Onion Creek, and print the number of
# features in the output.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 8. Matching a pattern: `LIKE`

The `LIKE` operator, used in place of `=`, matches part of a string with a **wildcard**, a character that stands for other characters. In SQL, `%` stands for any number of characters, including none. ArcPy list functions use `*` as their wildcard, but SQL does not.

| Pattern | Matches values that |
| --- | --- |
| `'K%'` | start with K |
| `'%WOOD'` | end with WOOD |
| `'%OAK%'` | contain OAK anywhere |

`LIKE` is case sensitive too, so `'%oak%'` does not match `OAK`.

In [ ]:
# Example 1 - LIKE matches part of a string

# % after the K matches any characters, so this matches every street name
# that starts with K
like_clause = "STREET_NAM LIKE 'K%'"
print(f"like_clause: {like_clause}")

# Write only the streets that match the pattern to a new feature class
like_result = arcpy.analysis.Select(streets_fc, "k_streets", like_clause)

# Count the features in the output
like_count = int(arcpy.management.GetCount(like_result)[0])
print(f"Streets that start with K: {like_count}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 8a] Use the Select tool and an appropriate where clause to select all
# hospitals in the hospitals feature class with the word Cancer anywhere in
# their name, and print the number of features in the output.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 9. Missing values: `IS NULL` and `IS NOT NULL`

In Python, `None` means no value. In a table, a field with no value for a record is **null**. A null is not zero and not an empty string `''`: it means the value is missing. To find nulls, SQL uses the `NULL` keyword, always after `IS` or `IS NOT`:

* `IS NULL` matches records with no value in the field.
* `IS NOT NULL` matches records with any value in the field.

`= NULL` does not find nulls, because a null is not equal to anything.

In [ ]:
# Example 1 - IS NULL matches records with no value

# IS NULL matches streets with no speed limit stored
null_clause = "SPEED_LIMI IS NULL"
print(f"null_clause: {null_clause}")

# Write only the streets with no speed limit to a new feature class
null_result = arcpy.analysis.Select(streets_fc, "no_speed", null_clause)

# Count the features in the output
null_count = int(arcpy.management.GetCount(null_result)[0])
print(f"Streets with no speed limit: {null_count}")

In [ ]:
# Example 2 - IS NOT NULL matches records with any value

# IS NOT NULL matches streets with any speed limit stored
not_null_clause = "SPEED_LIMI IS NOT NULL"
print(f"not_null_clause: {not_null_clause}")

# Write only the streets with a speed limit to a new feature class
not_null_result = arcpy.analysis.Select(
    streets_fc, "has_speed", not_null_clause
)

# Count the features in the output
not_null_count = int(arcpy.management.GetCount(not_null_result)[0])
print(f"Streets with a speed limit: {not_null_count}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 9a] Use the Select tool and an appropriate where clause to select all
# restrooms in the restrooms feature class with no value in the TYPE field,
# and print the number of features in the output.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 10. Building where clauses with f-strings

So far, every field name and value has been typed directly into the clause. In a script, they usually come from variables instead: a value read from a file, a field name from `arcpy.ListFields()`, or each item in a list. You have used f-strings to build strings from variables since `strings.ipynb`, and they build where clauses the same way.

The quoting rules from Section 4 still apply inside the f-string:

* The f-string itself goes inside double quotes.
* A string value goes inside single quotes inside the f-string: `'{street_type}'`.
* A number goes inside the braces with no quotes: `{speed_limit}`.

Print every clause you build before using it. Printing shows the SQL exactly as the tool receives it, with the variables replaced by their values.

In [ ]:
# Example 1 - Build where clauses from variables

# Variables store a field name, a string value, and a number
type_field = "STREET_TYP"
street_type = "DR"
speed_limit = 45

# The value is a string, so its braces are inside single quotes
type_clause = f"{type_field} = '{street_type}'"
print(f"type_clause: {type_clause}")

# The value is a number, so its braces are not inside quotes
speed_clause = f"SPEED_LIMI >= {speed_limit}"
print(f"speed_clause: {speed_clause}")

Building the clause from variables lets a loop run the same tool for every item in a list, without typing a new clause for each one.

In [ ]:
# Example 2 - Build a where clause for each item in a list

# A list of street types
street_types = ["DR", "LN", "CT"]

# We will use variable streets_fc from Section 3
for one_type in street_types:
    # Build the clause for the street type in this iteration
    one_clause = f"STREET_TYP = '{one_type}'"
    print(f"one_clause: {one_clause}")

    # Build an output name from the street type, such as streets_DR
    out_name = f"streets_{one_type}"

    # Select the matching streets and print the count
    one_result = arcpy.analysis.Select(streets_fc, out_name, one_clause)
    print(f"\tCount: {arcpy.management.GetCount(one_result)[0]}")

Older code, many Esri examples written before ArcGIS Pro, and a lot of AI-generated code put double quotes around field names, such as `"STREET_TYP" = 'LN'`. Those double quotes are **field delimiters**, characters that mark where a field name starts and ends. Field delimiters are not needed in ArcGIS Pro, so this notebook does not use them. You may also see the `arcpy.AddFieldDelimiters()` function, which Esri now lists as a legacy function.

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 10a] Use a for loop, an f-string, and the Select tool to select the
# facilities of each type in a list of two or three FACILITY_T values of
# your choice, and print each where clause with the number of features in
# its output.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this notebook, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

*End of notebook.*